In [1]:
import torch
import torch.nn as nn

_ = torch.manual_seed(26)

In [2]:
batch_size = 64
sequence_len = 64
embedding_size = 32
head_count = 4
layer_count = 4
steps = 5000

In [3]:
# input & output initialization
with open(file="inputs-shakespeare.txt", mode="r", encoding="utf-8") as f:
    data = f.read()

chars = sorted(set(data))
vocab_size = len(chars)
ctoi = {char: i for i, char in enumerate(iterable=chars)}
itoc = dict(enumerate(iterable=chars))

split = int(len(data) * 0.9)
train, val = data[:split], data[split:]

x_train = torch.tensor([ctoi[ch] for ch in train[:-1]], dtype=torch.long).unfold(dimension=0, size=sequence_len, step=1)
y_train = torch.tensor([ctoi[ch] for ch in train[1:]], dtype=torch.long).unfold(dimension=0, size=sequence_len, step=1)

x_val = torch.tensor([ctoi[ch] for ch in val[:-1]], dtype=torch.long).unfold(dimension=0, size=sequence_len, step=sequence_len)
y_val = torch.tensor([ctoi[ch] for ch in val[1:]], dtype=torch.long).unfold(dimension=0, size=sequence_len, step=sequence_len)

In [4]:
# module declaration
class TokenAndPositionEmbedding(nn.Module):
    def __init__(self, vocab_size, embedding_size, sequence_len):
        super().__init__()

        self.token_embedding = nn.Embedding(num_embeddings=vocab_size, embedding_dim=embedding_size)
        self.position_embedding = nn.Embedding(num_embeddings=sequence_len, embedding_dim=embedding_size)

    def forward(self, x):
        return self.token_embedding(x) + self.position_embedding(torch.arange(x.size(1)))

class SelfAttention(nn.Module):
    def __init__(self, embedding_size, head_size, sequence_len):
        super().__init__()

        self.head_size = head_size
        self.query = nn.Linear(in_features=embedding_size, out_features=head_size, bias=False)
        self.key = nn.Linear(in_features=embedding_size, out_features=head_size, bias=False)
        self.value = nn.Linear(in_features=embedding_size, out_features=head_size, bias=False)

        tril = torch.tril(torch.ones(size=(sequence_len, sequence_len), dtype=torch.bool))
        self.register_buffer(name="mask", tensor=tril, persistent=False)

    def forward(self, x):
        query = self.query(x)
        key = self.key(x)
        value = self.value(x)

        T = x.size(1)

        scores = query @ key.transpose(dim0=-2, dim1=-1) * self.head_size ** -0.5
        scores = scores.masked_fill(mask=~self.mask[:T, :T], value=float("-inf"))
        weights = torch.softmax(input=scores, dim=-1)

        return weights @ value

class MultiHeadAttention(nn.Module):
    def __init__(self, embedding_size, head_count, sequence_len):
        super().__init__()
        self.heads = nn.ModuleList(
            [SelfAttention(embedding_size=embedding_size, head_size=embedding_size // head_count, sequence_len=sequence_len) for _ in range(head_count)]
        )
        self.projection = nn.Linear(in_features=embedding_size, out_features=embedding_size)

    def forward(self, x):
        head_outputs = []
        for head in self.heads:
            head_output = head(x)
            head_outputs.append(head_output)
        return self.projection(torch.cat(head_outputs, dim=-1))

class FeedForward(nn.Module):
    def __init__(self, embedding_size):
        super().__init__()
        self.layers = nn.Sequential(
            nn.Linear(in_features=embedding_size, out_features=4 * embedding_size),
            nn.ReLU(),
            nn.Linear(in_features=4 * embedding_size, out_features=embedding_size),
        )
        
    def forward(self, x):
        return self.layers(x)

class Block(nn.Module):
    def __init__(self, embedding_size, head_count, sequence_len):
        super().__init__()
        self.attention = MultiHeadAttention(embedding_size=embedding_size, head_count=head_count, sequence_len=sequence_len)
        self.layer_norm1 = nn.LayerNorm(normalized_shape=embedding_size)
        self.feed_forward = FeedForward(embedding_size=embedding_size)
        self.layer_norm2 = nn.LayerNorm(normalized_shape=embedding_size)

    def forward(self, x):
        x = x + self.attention(self.layer_norm1(x))
        x = x + self.feed_forward(self.layer_norm2(x))
        return x

In [5]:
# model declaration
class Transformer(nn.Module):
    def __init__(
        self, vocab_size, embedding_size, sequence_len, head_count, layer_count
    ):
        super().__init__()
        self.layers = nn.Sequential(
            TokenAndPositionEmbedding(
                vocab_size=vocab_size,
                embedding_size=embedding_size,
                sequence_len=sequence_len,
            ),
            nn.Sequential(
                *[
                    Block(
                        embedding_size=embedding_size,
                        head_count=head_count,
                        sequence_len=sequence_len,
                    )
                    for _ in range(layer_count)
                ]
            ),
            nn.LayerNorm(normalized_shape=embedding_size),
            nn.Linear(in_features=embedding_size, out_features=vocab_size),
        )

    def forward(self, x):
        return self.layers(x)

In [6]:
# model initialization
model = Transformer(
    vocab_size=vocab_size,
    embedding_size=embedding_size,
    sequence_len=sequence_len,
    head_count=head_count,
    layer_count=layer_count,
)
print(f"parameters: {sum(param.numel() for param in model.parameters())}")
# optimizer initialization
optimizer = torch.optim.AdamW(params=model.parameters(), lr=3e-3)

parameters: 56769


In [7]:
# training
for step in range(steps):
    optimizer.zero_grad()

    batch = torch.randint(low=0, high=len(x_train), size=(batch_size,))
    x_batch = x_train[batch]
    y_batch = y_train[batch]
    train_logits = model(x_batch)

    train_loss = nn.functional.cross_entropy(input=train_logits.view(-1, vocab_size), target=y_batch.view(-1))

    if step % 1000 == 0 or step == steps - 1:
        model.eval()
        with torch.no_grad():
            val_logits = model(x_val)
            val_loss = nn.functional.cross_entropy(input=val_logits.view(-1, vocab_size), target=y_val.view(-1))
            print(f"Step {step}: train loss = {train_loss.item()}, val loss = {val_loss.item()}")
        model.train()
        
    train_loss.backward()
    optimizer.step()

Step 0: train loss = 4.369325160980225, val loss = 4.361491680145264


Step 1000: train loss = 1.935502052307129, val loss = 2.003584384918213


Step 2000: train loss = 1.7318994998931885, val loss = 1.8975536823272705


Step 3000: train loss = 1.6868187189102173, val loss = 1.83156418800354


Step 4000: train loss = 1.6120803356170654, val loss = 1.7971059083938599


Step 4999: train loss = 1.5448640584945679, val loss = 1.7902778387069702


In [8]:
# prediction
model.eval()
with torch.inference_mode():
    output = ['R', 'O', 'M', 'E', 'O', ':', '\n']
    max_output_length = 1000

    i = torch.tensor([[ctoi[ch] for ch in output]], dtype=torch.long)

    while len(output) < max_output_length:
        logits = model(i)

        P = torch.softmax(input=logits[0, -1, :], dim=0)
        j = torch.multinomial(input=P, num_samples=1, replacement=True).item()
        output.append(itoc[j])

        i = torch.tensor([[ctoi[ch] for ch in output[-sequence_len:]]], dtype=torch.long)

In [9]:
print(''.join(output))

ROMEO:
What seh hath though a pardomies cupurt
In may shave thou as strike Gode hasted on dearet
And I gity the blest of them.

WARWICK:
What father revengents, and plel painson,
And facumousaty a king he countryment, angaim,
if dom to cart is with time blow pity;
Which in the peechention! We I'll is you;
Where did a' strong a slobe should optake.

MENENIUS:
And, Lord Mon dear:
Now you die, CAPULET:
I may,
Profith, but, for law!'ling he sove's sew
Have othe the coose now, we the loves!

For GLOTHWard defe,
With you grazes aloovedy: throw worthy, subje.

LORD LARD:
Though thy waith termes dity
harrd, husbank, for son at much be shall do the
as much it rown not much my tracks him,
To the piace Jicity high comothed me?

FRIARST:
Ay, fargest weated sliends.
She both to noath so, it on, my hath find:
She is
TRetion, thou ever ble mave dever ware shall,
Appainit looke apponing, us give a when?

LLEOge;
Or Greathe it, to, more time it the done
Hose ropins.

CHESCANT:
Epong, be of lange my hav